# Week 10: LangGraph in Practice + LangSmith & Langfuse

**Goal:** Build LangGraph workflow patterns and connect them to real observability dashboards.

**What you'll build:**
- Chaining, routing, and evaluator-optimizer workflows
- Human-in-the-loop approval flow
- Full trace visibility in LangSmith and Langfuse
- Evaluation datasets with LLM-as-judge scoring

**Prerequisites:** Complete the setup cells before running exercises.

## Setup

In [ ]:
!pip install langgraph langchain-openai langsmith langfuse python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Required: OpenAI API key
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

# Optional but recommended for this week:
# LangSmith: LANGCHAIN_API_KEY + LANGCHAIN_TRACING_V2=true
# Langfuse:  LANGFUSE_PUBLIC_KEY + LANGFUSE_SECRET_KEY

langsmith_enabled = bool(os.getenv("LANGCHAIN_API_KEY") and os.getenv("LANGCHAIN_TRACING_V2"))
langfuse_enabled  = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))

print(f"OpenAI: ready")
print(f"LangSmith: {'enabled' if langsmith_enabled else 'not configured (optional)'}")
print(f"Langfuse:  {'enabled' if langfuse_enabled  else 'not configured (optional)'}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Literal
import operator, json, re

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imports OK")

---
## Part 1: LangGraph Workflow Patterns

Build three patterns from scratch.

### Exercise 1: Chaining (A → B → C)

Build a pipeline that **summarizes** text, then **translates** the summary to French, then **formats** it.

```
START → summarize → translate → format → END
```

In [ ]:
class ChainState(TypedDict):
    text: str
    summary: str
    translated: str
    final: str

# TODO: Define three nodes: summarize, translate_to_french, format_output
# Each node takes state and returns a dict with updated keys

def summarize(state: ChainState) -> dict:
    # Your code here
    pass

def translate_to_french(state: ChainState) -> dict:
    # Your code here
    pass

def format_output(state: ChainState) -> dict:
    # Your code here
    pass

# TODO: Build the graph, compile, and test it
chain_app = None  # replace with compiled graph

# Test
if chain_app:
    result = chain_app.invoke({"text": "The mitochondria is the powerhouse of the cell, producing ATP through cellular respiration."})
    print(result["final"])

<details>
<summary>💡 Hint</summary>

Each node function must return a **dict** with only the keys it changes. Use `llm.invoke([SystemMessage(...), HumanMessage(state["key"])])` and return `{"new_key": response.content}`.

For the graph:
```python
builder = StateGraph(ChainState)
builder.add_node("summarize", summarize)
# ... add remaining nodes
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "translate")
# ... add remaining edges
chain_app = builder.compile()
```
</details>

<details>
<summary>✅ Solution</summary>

In [ ]:
def summarize(state: ChainState) -> dict:
    r = llm.invoke([SystemMessage("Summarize in one sentence."), HumanMessage(state["text"])])
    return {"summary": r.content}

def translate_to_french(state: ChainState) -> dict:
    r = llm.invoke([SystemMessage("Translate to French."), HumanMessage(state["summary"])])
    return {"translated": r.content}

def format_output(state: ChainState) -> dict:
    return {"final": f"FR: {state['translated']}"}

chain_builder = StateGraph(ChainState)
chain_builder.add_node("summarize", summarize)
chain_builder.add_node("translate", translate_to_french)
chain_builder.add_node("format", format_output)
chain_builder.add_edge(START, "summarize")
chain_builder.add_edge("summarize", "translate")
chain_builder.add_edge("translate", "format")
chain_builder.add_edge("format", END)
chain_app = chain_builder.compile()

result = chain_app.invoke({"text": "The mitochondria is the powerhouse of the cell, producing ATP through cellular respiration."})
print("Summary:", result["summary"])
print("Final:  ", result["final"])

</details>

### Exercise 2: Routing

Build a graph that **classifies** a question as `math`, `science`, or `general`, then routes it to the right expert node.

```
START → classify → [math_expert | science_expert | general_assistant] → END
```

In [ ]:
class RouteState(TypedDict):
    question: str
    category: str
    answer: str

# TODO: implement classify(), route(), and three expert nodes
# route() must return a Literal string matching a node name

def classify(state: RouteState) -> dict:
    pass

def route(state: RouteState) -> Literal["math_expert", "science_expert", "general_assistant"]:
    pass

# TODO: build and test
route_app = None

if route_app:
    for q in ["What is 144 divided by 12?", "Why does ice float on water?", "Tell me a fun fact"]:
        r = route_app.invoke({"question": q})
        print(f"[{r['category']:8}] {r['answer'][:70]}")

<details>
<summary>💡 Hint</summary>

For routing:
```python
builder.add_conditional_edges("classify", route)
# LangGraph maps the return string to a node name automatically
```

Make sure `route()` returns **exactly** the node name as a string.
</details>

<details>
<summary>✅ Solution</summary>

In [ ]:
def classify(state: RouteState) -> dict:
    r = llm.invoke([
        SystemMessage('Classify the question. Reply with exactly one word: "math", "science", or "general"'),
        HumanMessage(state["question"])
    ])
    return {"category": r.content.strip().lower()}

def route(state: RouteState) -> Literal["math_expert", "science_expert", "general_assistant"]:
    c = state["category"]
    if "math" in c:    return "math_expert"
    if "science" in c: return "science_expert"
    return "general_assistant"

def math_expert(state: RouteState) -> dict:
    r = llm.invoke([SystemMessage("You are a math expert."), HumanMessage(state["question"])])
    return {"answer": r.content}

def science_expert(state: RouteState) -> dict:
    r = llm.invoke([SystemMessage("You are a science expert."), HumanMessage(state["question"])])
    return {"answer": r.content}

def general_assistant(state: RouteState) -> dict:
    r = llm.invoke([HumanMessage(state["question"])])
    return {"answer": r.content}

rb = StateGraph(RouteState)
for name, fn in [("classify", classify), ("math_expert", math_expert),
                  ("science_expert", science_expert), ("general_assistant", general_assistant)]:
    rb.add_node(name, fn)
rb.add_edge(START, "classify")
rb.add_conditional_edges("classify", route)
for n in ["math_expert", "science_expert", "general_assistant"]:
    rb.add_edge(n, END)
route_app = rb.compile()

for q in ["What is 144 divided by 12?", "Why does ice float on water?", "Tell me a fun fact"]:
    r = route_app.invoke({"question": q})
    print(f"[{r['category']:8}] {r['answer'][:70]}")

</details>

### Exercise 3: Evaluator-Optimizer Loop

Build a loop that generates a draft, judges its quality (0–1), and keeps regenerating until score ≥ 0.8 or 3 iterations.

```
START → generate → evaluate → [generate (loop) | accept] → END
```

In [ ]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class EvalState(TypedDict):
    task: str
    draft: str
    feedback: str
    score: float
    iterations: int
    final: str

THRESHOLD = 0.8
MAX_ITER  = 3

# TODO: implement generate(), evaluate(), should_continue(), accept()
# evaluate() should return {"score": float, "feedback": str} parsed from LLM JSON output
# should_continue() returns "accept" or "generate"

eval_app = None  # replace with compiled graph

if eval_app:
    config = {"configurable": {"thread_id": "eval-lab"}}
    result = eval_app.invoke(
        {"task": "Write a 2-sentence pitch for an AI observability tool.", "iterations": 0},
        config=config
    )
    print(f"Iterations: {result['iterations']} | Score: {result['score']:.2f}")
    print(f"Output: {result['final']}")

<details>
<summary>💡 Hint</summary>

For the judge, prompt it to respond with JSON and parse with `re.search(r'\{.*\}', content, re.DOTALL)`.

For the loop:
```python
builder.add_edge("accept", END)
builder.add_conditional_edges("evaluate", should_continue)
# should_continue maps "generate" -> "generate" node, "accept" -> "accept" node
```

Compile with `checkpointer=MemorySaver()` to inspect state history.
</details>

<details>
<summary>✅ Solution</summary>

In [ ]:
gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def generate(state: EvalState) -> dict:
    prompt = state["task"]
    if state.get("feedback"):
        prompt += f"\n\nAddress this feedback:\n{state['feedback']}"
    r = gen_llm.invoke([SystemMessage("You are a skilled writer."), HumanMessage(prompt)])
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

def evaluate(state: EvalState) -> dict:
    r = judge_llm.invoke([
        SystemMessage('Rate this output 0.0-1.0. Respond ONLY as JSON: {"score": 0.7, "feedback": "..."}'),
        HumanMessage(f"Task: {state['task']}\n\nOutput:\n{state['draft']}")
    ])
    match = re.search(r'\{.*\}', r.content, re.DOTALL)
    data = json.loads(match.group()) if match else {"score": 0.5, "feedback": "no feedback"}
    return {"score": data["score"], "feedback": data["feedback"]}

def should_continue(state: EvalState) -> Literal["generate", "accept"]:
    return "accept" if state["score"] >= THRESHOLD or state["iterations"] >= MAX_ITER else "generate"

def accept(state: EvalState) -> dict:
    return {"final": state["draft"]}

eb = StateGraph(EvalState)
eb.add_node("generate", generate)
eb.add_node("evaluate", evaluate)
eb.add_node("accept", accept)
eb.add_edge(START, "generate")
eb.add_edge("generate", "evaluate")
eb.add_conditional_edges("evaluate", should_continue)
eb.add_edge("accept", END)
eval_app = eb.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "eval-lab"}}
result = eval_app.invoke(
    {"task": "Write a 2-sentence pitch for an AI observability tool.", "iterations": 0},
    config=config
)
print(f"Iterations: {result['iterations']} | Score: {result['score']:.2f}")
print(f"Output: {result['final']}")

# Show checkpoints
history = list(eval_app.get_state_history(config))
print(f"\n{len(history)} checkpoints:")
for step in reversed(history):
    print(f"  next={step.next} | score={step.values.get('score', '-')}")

</details>

### Exercise 4: Human-in-the-Loop

Build an agent that **proposes** an action, **pauses** for human approval using `interrupt()`, then **executes or rejects** based on the response.

```
START → plan → [interrupt] → review → execute OR reject → END
```

In [ ]:
from langgraph.types import interrupt

class ReviewState(TypedDict):
    messages: Annotated[list, operator.add]
    draft_action: str
    approved: bool

# TODO: implement plan_action, human_review (using interrupt()), execute_action, reject_action
# Build graph with interrupt_before=["review"]
# Test: invoke → get snapshot → update_state → invoke(None) to resume

hitl_app = None  # replace with compiled graph

<details>
<summary>💡 Hint</summary>

The key pattern:
```python
# Compile with interrupt_before
app = builder.compile(checkpointer=MemorySaver(), interrupt_before=["review"])

# First invoke — pauses before "review"
app.invoke(initial_state, config)

# Inject human decision
app.update_state(config, {"approved": True}, as_node="review")

# Resume (pass None as input)
app.invoke(None, config)
```

Inside `human_review`, use `interrupt({"question": "...", "action": state['draft_action']})` — the return value is whatever was passed to `update_state`.
</details>

<details>
<summary>✅ Solution</summary>

In [ ]:
def plan_action(state: ReviewState) -> dict:
    r = llm.invoke(state["messages"] + [HumanMessage("What specific action should we take? Be concrete.")])
    return {"messages": [AIMessage(r.content)], "draft_action": r.content}

def human_review(state: ReviewState) -> dict:
    decision = interrupt({"question": "Approve?", "action": state["draft_action"]})
    approved = str(decision).lower() in ("yes", "y", "approve")
    return {"approved": approved, "messages": [HumanMessage(f"Decision: {'approved' if approved else 'rejected'}")]}

def execute_action(state: ReviewState) -> dict:
    r = llm.invoke([HumanMessage(f"Confirm execution of: {state['draft_action']}")])
    return {"messages": [AIMessage(f"Executed: {r.content[:80]}")]}

def reject_action(state: ReviewState) -> dict:
    return {"messages": [AIMessage("Action rejected — no changes made.")]}

hb = StateGraph(ReviewState)
hb.add_node("plan", plan_action)
hb.add_node("review", human_review)
hb.add_node("execute", execute_action)
hb.add_node("reject", reject_action)
hb.add_edge(START, "plan")
hb.add_edge("plan", "review")
hb.add_conditional_edges("review", lambda s: "execute" if s["approved"] else "reject")
hb.add_edge("execute", END)
hb.add_edge("reject", END)
hitl_app = hb.compile(checkpointer=MemorySaver(), interrupt_before=["review"])

config = {"configurable": {"thread_id": "hitl-demo"}}

# Run until interrupt
hitl_app.invoke(
    {"messages": [HumanMessage("Archive all log files older than 90 days.")], "approved": False},
    config=config
)
snapshot = hitl_app.get_state(config)
print("Paused before:", snapshot.next)
print("Proposed:", snapshot.values["draft_action"][:120])

# Approve and resume
hitl_app.update_state(config, {"approved": True}, as_node="review")
final = hitl_app.invoke(None, config)
for msg in final["messages"][-2:]:
    print(f"[{type(msg).__name__}]: {msg.content[:80]}")

</details>

---
## Part 2: LangSmith Observability

**Setup first:**
```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY=ls-...
export LANGCHAIN_PROJECT=week-10-lab
```
Then restart the kernel so environment variables are picked up.

In [ ]:
import os
tracing = os.getenv("LANGCHAIN_TRACING_V2") == "true" and bool(os.getenv("LANGCHAIN_API_KEY"))
print(f"LangSmith tracing: {'ON' if tracing else 'OFF — set env vars and restart kernel'}")

### Exercise 5: Traced RAG Pipeline

Add `@traceable` spans to a RAG pipeline so each step appears as a named span in LangSmith.

In [ ]:
from langsmith import traceable

# TODO: Add @traceable(name="retrieval") to fake_retrieval
# TODO: Add @traceable(name="rag_pipeline") to rag_answer

def fake_retrieval(query: str) -> list:
    return [f"Doc 1 about {query}: relevant content", f"Doc 2 about {query}: more context"]

def rag_answer(question: str) -> str:
    docs = fake_retrieval(question)
    context = "\n".join(docs)
    r = llm.invoke([HumanMessage(f"Context:\n{context}\n\nQuestion: {question}")])
    return r.content

# Test — check smith.langchain.com for the trace
answer = rag_answer("What is LangGraph?")
print(f"Answer: {answer[:100]}")
print("-> Open smith.langchain.com and look for a 'rag_pipeline' trace")

<details>
<summary>✅ Solution</summary>

In [ ]:
@traceable(name="retrieval")
def fake_retrieval(query: str) -> list:
    return [f"Doc 1 about {query}: relevant content", f"Doc 2 about {query}: more context"]

@traceable(name="rag_pipeline")
def rag_answer(question: str) -> str:
    docs = fake_retrieval(question)
    context = "\n".join(docs)
    r = llm.invoke([HumanMessage(f"Context:\n{context}\n\nQuestion: {question}")])
    return r.content

answer = rag_answer("What is LangGraph?")
print(f"Answer: {answer[:100]}")

</details>

### Exercise 6: LangSmith Evaluation Dataset

Create a dataset with 3 Q&A pairs, run your `rag_answer` function against it, and score results with an LLM-as-judge evaluator.

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

# TODO:
# 1. Create or reuse a dataset named "week10-lab-eval"
# 2. Add 3 examples with {"question": ...} inputs and {"answer": ...} expected outputs
# 3. Define a correctness_evaluator(run, example) -> dict
# 4. Call evaluate() and print results

print("Implement the evaluation pipeline above")

<details>
<summary>✅ Solution</summary>

In [ ]:
dataset_name = "week10-lab-eval"
existing = [d.name for d in client.list_datasets()]

if dataset_name not in existing:
    ds = client.create_dataset(dataset_name)
    client.create_examples(
        inputs=[
            {"question": "What is LangGraph?"},
            {"question": "What is LangSmith?"},
            {"question": "How does RAG work?"},
        ],
        outputs=[
            {"answer": "LangGraph is a framework for building stateful agent workflows as directed graphs"},
            {"answer": "LangSmith is an observability and evaluation platform for LLM applications"},
            {"answer": "RAG retrieves relevant documents and injects them as context for the LLM"},
        ],
        dataset_id=ds.id
    )
    print(f"Created dataset: {dataset_name}")

def correctness_evaluator(run, example):
    r = llm.invoke([HumanMessage(
        f"Rate correctness 0.0-1.0. Expected: '{example.outputs['answer']}'. "
        f"Got: '{run.outputs['output']}'. Reply with float only."
    )])
    try:
        score = float(r.content.strip())
    except:
        score = 0.5
    return {"key": "correctness", "score": min(1.0, max(0.0, score))}

results = evaluate(
    lambda inputs: {"output": rag_answer(inputs["question"])},
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="week10-lab",
    max_concurrency=1
)
print("Evaluation complete — see smith.langchain.com/datasets for scores")

</details>

---
## Part 3: Langfuse Observability

**Setup:**
```bash
# Self-hosted (recommended):
git clone https://github.com/langfuse/langfuse && cd langfuse
cp .env.langfuse.example .env  # edit NEXTAUTH_SECRET and SALT
docker compose up -d
# Create API keys at http://localhost:3000 -> Settings

export LANGFUSE_PUBLIC_KEY=pk-lf-...
export LANGFUSE_SECRET_KEY=sk-lf-...
export LANGFUSE_HOST=http://localhost:3000
```

In [ ]:
lf_ready = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))
print(f"Langfuse: {'ready' if lf_ready else 'not configured — follow setup above'}")

### Exercise 7: Langfuse @observe Tracing

Use the `@observe` decorator to trace the same RAG pipeline in Langfuse. Add a score to the root trace.

In [ ]:
from langfuse.decorators import observe, langfuse_context
from langfuse import Langfuse

langfuse = Langfuse()

# TODO: Add @observe(name="retrieval") to lf_retrieval
# TODO: Add @observe(name="rag_pipeline") to lf_rag_answer
# TODO: Inside lf_rag_answer, call langfuse_context.score_current_trace(name="relevance", value=0.85)

def lf_retrieval(query: str) -> list:
    return [f"Doc 1 about {query}", f"Doc 2 about {query}"]

def lf_rag_answer(question: str) -> str:
    docs = lf_retrieval(question)
    context = "\n".join(docs)
    r = llm.invoke([HumanMessage(f"Context: {context}\n\nQ: {question}")])
    return r.content

answer = lf_rag_answer("What is checkpointing in LangGraph?")
print(f"Answer: {answer[:100]}")
langfuse.flush()
print("-> Check your Langfuse dashboard for the trace!")

<details>
<summary>✅ Solution</summary>

In [ ]:
@observe(name="retrieval")
def lf_retrieval(query: str) -> list:
    return [f"Doc 1 about {query}", f"Doc 2 about {query}"]

@observe(name="rag_pipeline")
def lf_rag_answer(question: str) -> str:
    docs = lf_retrieval(question)
    context = "\n".join(docs)
    r = llm.invoke([HumanMessage(f"Context: {context}\n\nQ: {question}")])
    langfuse_context.score_current_trace(
        name="relevance",
        value=0.85,
        comment="heuristic score"
    )
    return r.content

answer = lf_rag_answer("What is checkpointing in LangGraph?")
print(f"Answer: {answer[:100]}")
langfuse.flush()

</details>

### Exercise 8: LangGraph + Langfuse CallbackHandler

Trace a LangGraph agent run through Langfuse using `CallbackHandler`. Attach a manual score after the run.

In [ ]:
from langfuse.callback import CallbackHandler

# Reuse the chain_app from Exercise 1 (or rebuild a simple one)
# TODO:
# 1. Create a CallbackHandler()
# 2. Pass it via config={"callbacks": [handler]} to app.invoke()
# 3. After the run, attach a score using langfuse.score(trace_id=handler.get_trace_id(), ...)
# 4. Call langfuse.flush()

print("Implement above")

<details>
<summary>✅ Solution</summary>

In [ ]:
handler = CallbackHandler()  # reads LANGFUSE_* env vars

result = chain_app.invoke(
    {"text": "The James Webb Space Telescope can observe galaxies formed just after the Big Bang."},
    config={"callbacks": [handler]}
)
print("Result:", result["final"])

# Attach a score to the trace
langfuse.score(
    trace_id=handler.get_trace_id(),
    name="quality",
    value=0.9,
    comment="translation looked good"
)
langfuse.flush()
print("-> Check Langfuse for the scored trace!")

</details>

### Exercise 9: Langfuse Dataset Evaluation

Create a Langfuse dataset, run your pipeline against each item, and score each result with an LLM judge.

In [ ]:
# TODO:
# 1. Create/get a dataset named "week10-langfuse-eval" with 3 LangGraph Q&A items
# 2. For each item, run lf_rag_answer() inside item.observe(run_name=...)
# 3. Score each trace with langfuse.score() using an LLM-as-judge
# 4. langfuse.flush()

print("Implement above")

<details>
<summary>✅ Solution</summary>

In [ ]:
ds_name = "week10-langfuse-eval"
try:
    dataset = langfuse.get_dataset(ds_name)
except Exception:
    langfuse.create_dataset(ds_name)
    items = [
        ("What is a StateGraph?",   "A StateGraph defines the graph structure with typed state in LangGraph"),
        ("What is a reducer?",      "A reducer merges node output into existing state via Annotated type hints"),
        ("What does interrupt() do?", "interrupt() pauses graph execution waiting for human input"),
    ]
    for q, a in items:
        langfuse.create_dataset_item(dataset_name=ds_name, input={"question": q}, expected_output={"answer": a})
    dataset = langfuse.get_dataset(ds_name)
    print(f"Created dataset with {len(items)} items")

for item in dataset.items:
    q = item.input["question"]
    expected = item.expected_output["answer"]

    with item.observe(run_name="gpt-4o-mini-lab") as obs_handler:
        answer = lf_rag_answer(q)

        judge_r = llm.invoke([HumanMessage(
            f"Rate 0.0-1.0: Expected: {expected}\nGot: {answer}\nFloat only."
        )])
        try:
            score = float(judge_r.content.strip())
        except:
            score = 0.5

        langfuse.score(
            trace_id=obs_handler.get_trace_id(),
            name="semantic_match",
            value=min(1.0, max(0.0, score))
        )
        print(f"  Q: {q[:50]}... | Score: {score:.2f}")

langfuse.flush()
print("-> Check Langfuse > Datasets for scores!")

</details>

---
## Bonus: Dual Tracing

Trace the same LangGraph tool-calling agent through **both** LangSmith and Langfuse simultaneously.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

@tool
def count_words(text: str) -> int:
    """Count words in text."""
    return len(text.split())

@tool
def reverse_text(text: str) -> str:
    """Reverse a string."""
    return text[::-1]

tools = [count_words, reverse_text]
tool_llm = llm.bind_tools(tools)

def tool_agent(state: MessagesState) -> dict:
    return {"messages": [tool_llm.invoke(state["messages"])]}

def check_tools(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

tb = StateGraph(MessagesState)
tb.add_node("agent", tool_agent)
tb.add_node("tools", ToolNode(tools))
tb.add_edge(START, "agent")
tb.add_conditional_edges("agent", check_tools)
tb.add_edge("tools", "agent")
tool_app = tb.compile()

# Run with Langfuse callback (LangSmith is automatic via env vars)
lf_handler = CallbackHandler()

result = tool_app.invoke(
    {"messages": [HumanMessage("How many words in 'Hello world from LangGraph'? Then reverse that phrase.")]},
    config={"callbacks": [lf_handler]}
)

for msg in result["messages"]:
    content = msg.content or str(getattr(msg, 'tool_calls', ''))
    print(f"[{type(msg).__name__}]: {str(content)[:80]}")

lf_handler.langfuse.flush()
print("\n-> Check both smith.langchain.com AND Langfuse dashboard!")

---
## Summary

**What you built this week:**

| Pattern | Key concept |
|---------|------------|
| Chaining | Sequential nodes, state flows through |
| Routing | `add_conditional_edges` + routing function |
| Evaluator-optimizer | Looping edges + score threshold |
| Human-in-the-loop | `interrupt()` + `update_state()` + `invoke(None)` |

| Observability | Key setup |
|--------------|----------|
| LangSmith | `LANGCHAIN_TRACING_V2=true` — automatic for all LangGraph |
| Langfuse | `@observe` or `CallbackHandler` — works with any stack |
| Both | Pass `CallbackHandler` to `config["callbacks"]` — LangSmith is still auto |

**Next:** Week 11 — Advanced LangGraph (orchestrator-worker, deep agents) + Vision/Audio APIs.